# Retrieval Evaluation: Precision@k, Recall@k, NDCG@k

The hybrid RAG notebooks (`2_hybrid_rag`) show that dense, sparse, and RRF-fused retrieval return *different* ranked lists for the same query - but "looks reasonable" isn't a metric. This notebook scores each retriever against a small **labeled** query set (query -> which chunks are actually relevant) using three standard retrieval metrics:

- **Precision@k** - of the top-k chunks returned, what fraction are relevant? (measures noise in the results)
- **Recall@k** - of all relevant chunks that exist, what fraction did the top-k retrieve? (measures how much was missed)
- **NDCG@k** (Normalized Discounted Cumulative Gain) - like recall, but rewards relevant chunks for appearing *higher* in the ranking, not just being present

```
labeled queries --> [ dense / sparse / hybrid retrieval ] --> ranked chunks --> [ compare vs. ground truth ] --> P@k, R@k, NDCG@k
```

We reuse the exact same chunks, embeddings, and BM25 index built in `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb` and queried in `2_hybrid_rag/2_hybrid_search_rrf.ipynb` - this notebook only adds the scoring layer on top.

Prerequisites: run `1_standard_rag/4_ocr_chunk_store_pgvector.ipynb` first (populates `rag_documents` and builds `bm25_chunks_idx`), with `pg_textsearch` installed (see `1_install_pgvector`).

## Dependencies

Same stack as the hybrid RAG notebooks - `langchain-postgres` + `langchain-openai` (dense retrieval), `psycopg` (BM25 SQL), `python-dotenv` - all already in `../requirements.txt`. No new packages; the metrics themselves are plain Python (`math.log2`).

```bash
pip install -r requirements.txt
```

## Configuration and connections

Same shared `day2skk/var.env` file as every other notebook, and the same `rag_documents` collection / `bm25_chunks_idx` index populated by notebook 4. We open a `psycopg` connection for BM25 SQL and a `PGVector` store for dense search, exactly as in `2_hybrid_rag/2_hybrid_search_rrf.ipynb`.

In [ ]:
import os
import psycopg
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_postgres import PGVector

load_dotenv("../var.env")   # shared env file at day2skk/var.env

PG_HOST = os.environ.get("PG_HOST", "pgvector")
PG_PORT = os.environ.get("PG_PORT", "5432")
PG_USER = os.environ.get("PG_USER", "raguser")
PG_PASSWORD = os.environ.get("PG_PASSWORD", "change-me-please")
PG_DB = os.environ.get("PG_DB", "ragdb")
COLLECTION_NAME = "rag_documents"   # collection ingested in 1_standard_rag/4
INDEX_NAME = "bm25_chunks_idx"      # BM25 index created in 1_standard_rag/4

# Embedding model: self-hosted Qwen3-Embedding-4B on vLLM. MUST match ingestion.
EMB_MODEL = os.environ.get("EMB_MODEL", "Qwen/Qwen3-Embedding-4B")
EMB_BASE_URL = os.environ.get("EMB_BASE_URL", "http://qwen3-emb-4b.default.svc.cluster.local/v1")
EMB_API_KEY = os.environ.get("EMB_API_KEY", "sk-dhU11z5FIjXQ8TLXpwDotpGetP21CQv3")

# psycopg connection for BM25 SQL
conn = psycopg.connect(
    f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASSWORD}"
)
conn.autocommit = True

# PGVector store for dense search (same embeddings + collection as notebook 4).
embeddings = OpenAIEmbeddings(
    model=EMB_MODEL,
    base_url=EMB_BASE_URL,
    api_key=EMB_API_KEY,
    check_embedding_ctx_length=False,
)
connection_url = f"postgresql+psycopg://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"
vector_store = PGVector(
    embeddings=embeddings,
    collection_name=COLLECTION_NAME,
    connection=connection_url,
    use_jsonb=True,
)

with conn.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_extension WHERE extname = 'pg_textsearch';")
    if cur.fetchone() is None:
        raise RuntimeError("pg_textsearch is not installed. See 1_install_pgvector.")
    cur.execute("SELECT 1 FROM pg_indexes WHERE indexname = %s;", (INDEX_NAME,))
    if cur.fetchone() is None:
        raise RuntimeError(f"Index {INDEX_NAME!r} not found. Run 1_standard_rag/4 first.")
    cur.execute("SELECT uuid FROM langchain_pg_collection WHERE name = %s;", (COLLECTION_NAME,))
    row = cur.fetchone()
    if row is None:
        raise RuntimeError(f"Collection {COLLECTION_NAME!r} not found. Run 1_standard_rag/4 first.")
    COLLECTION_ID = row[0]

print("connected; collection id:", COLLECTION_ID)

## Step 1: Retrievers that return chunk IDs

Metrics need a stable identifier per chunk, not raw text. Notebook 4 tagged every chunk with `chunk_index` metadata at ingestion time, so each retriever below returns a **ranked list of `chunk_index` values** instead of chunk text. `hybrid_search` fuses the dense and sparse rankings with the same Reciprocal Rank Fusion used in `2_hybrid_rag/2_hybrid_search_rrf.ipynb`, just keyed by `chunk_index` instead of text.

In [ ]:
def dense_search(query: str, k: int = 5) -> list[int]:
    docs = vector_store.similarity_search(query, k=k)
    return [d.metadata["chunk_index"] for d in docs]


def sparse_search(query: str, k: int = 5) -> list[int]:
    sql = (
        "SELECT (cmetadata->>'chunk_index')::int "
        "FROM langchain_pg_embedding "
        "WHERE collection_id = %(cid)s "
        "ORDER BY document <@> to_bm25query(%(q)s, %(idx)s) "   # ascending = best first
        "LIMIT %(k)s;"
    )
    with conn.cursor() as cur:
        cur.execute(sql, {"cid": COLLECTION_ID, "q": query, "idx": INDEX_NAME, "k": k})
        return [r[0] for r in cur.fetchall()]


def rrf_fuse(ranked_lists: list[list[int]], rrf_k: int = 60) -> list[tuple[int, float]]:
    scores: dict[int, float] = {}
    for ranked in ranked_lists:
        for rank, key in enumerate(ranked):
            scores[key] = scores.get(key, 0.0) + 1.0 / (rrf_k + rank + 1)
    return sorted(scores.items(), key=lambda kv: kv[1], reverse=True)


def hybrid_search(query: str, k_each: int = 5, rrf_k: int = 60) -> list[int]:
    dense = dense_search(query, k_each)
    sparse = sparse_search(query, k_each)
    fused = rrf_fuse([dense, sparse], rrf_k=rrf_k)
    return [chunk_index for chunk_index, _ in fused]


RETRIEVERS = {
    "dense": dense_search,
    "sparse": sparse_search,
    "hybrid": hybrid_search,
}
print("retrievers ready:", list(RETRIEVERS))

## Step 2: A small labeled evaluation set

This is the piece that can't be automated: for each query, a human (or a careful LLM-assisted pass, spot-checked by a human) decides which chunks actually answer it - the **ground truth**. `cloudeka.pdf` now chunks into 18 passages covering the platform overview, Prepaid/Postpaid, and each product (Deka Box, CDN, Claw, DNS, LLM, Metal, Notebook, SSL, Vault GPU, VPN, and `cldkctl`; see notebook 4 / `chunks.txt`). The set below covers most of those chunks, but is still small for illustration - a real evaluation set should have **dozens to hundreds** of queries sampled from actual user traffic, covering easy and hard cases, with agreed-upon relevance judgments.

In [ ]:
# Ground truth: query -> set of chunk_index values that correctly answer it.
EVAL_SET = [
    {"query": "What is Cloudeka?", "relevant": {0}},
    {"query": "What is the Prepaid project type used for?", "relevant": {1}},
    {"query": "What is the Postpaid project type used for?", "relevant": {2}},
    {"query": "What is Deka Box?", "relevant": {3}},
    {"query": "What is Deka CDN used for?", "relevant": {3}},
    {"query": "What is Deka Claw used for?", "relevant": {4}},
    {"query": "What is Deka DNS used for?", "relevant": {5}},
    {"query": "What are the Deka LLM model categories?", "relevant": {7, 8, 9, 10}},
    {"query": "What is the VLM model category used for?", "relevant": {9}},
    {"query": "What is Deka Metal used for?", "relevant": {12}},
    {"query": "What is Deka SSL?", "relevant": {13}},
    {"query": "What is Deka Vault GPU used for?", "relevant": {14}},
    {"query": "What protocols does Deka VPN support?", "relevant": {15}},
    {"query": "What is cldkctl used for?", "relevant": {16}},
]

K = 5   # cutoff for all metrics below
print(f"{len(EVAL_SET)} labeled queries, evaluating at k={K}")

## Step 3: Precision@k, Recall@k, NDCG@k

All three use **binary relevance** (a chunk is either relevant or not - no partial credit):

- **Precision@k** = (relevant chunks in top-k) / k - penalizes irrelevant chunks cluttering the results.
- **Recall@k** = (relevant chunks in top-k) / (total relevant chunks) - penalizes missing relevant chunks entirely.
- **DCG@k** = sum over top-k of `relevance / log2(rank + 1)` - relevant chunks count less the further down they appear.
- **NDCG@k** = DCG@k / IDCG@k, where IDCG is the DCG of the *ideal* ranking (all relevant chunks first). Normalizing to `[0, 1]` makes NDCG comparable across queries with different numbers of relevant chunks.

In [ ]:
import math


def precision_at_k(ranked: list[int], relevant: set[int], k: int) -> float:
    top_k = ranked[:k]
    if not top_k:
        return 0.0
    hits = sum(1 for c in top_k if c in relevant)
    return hits / len(top_k)


def recall_at_k(ranked: list[int], relevant: set[int], k: int) -> float:
    if not relevant:
        return 0.0
    hits = sum(1 for c in ranked[:k] if c in relevant)
    return hits / len(relevant)


def dcg_at_k(ranked: list[int], relevant: set[int], k: int) -> float:
    return sum(
        (1.0 if c in relevant else 0.0) / math.log2(rank + 1)
        for rank, c in enumerate(ranked[:k], start=2)   # rank starts at 2 so log2(rank) is defined
    )


def ndcg_at_k(ranked: list[int], relevant: set[int], k: int) -> float:
    dcg = dcg_at_k(ranked, relevant, k)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1.0 / math.log2(rank + 1) for rank in range(2, ideal_hits + 2))
    return dcg / idcg if idcg > 0 else 0.0


# sanity check: a ranking with every relevant chunk first should score 1.0 on all three
assert precision_at_k([0, 1], {0, 1}, 2) == 1.0
assert recall_at_k([0, 1], {0, 1}, 2) == 1.0
assert ndcg_at_k([0, 1], {0, 1}, 2) == 1.0
print("metric functions ready")

## Step 4: Score every retriever against the eval set

For each retriever (dense, sparse, hybrid), run every labeled query, score the ranked results against ground truth, and average the per-query metrics. This is exactly how you'd A/B two retrieval strategies - swap in a different `hybrid_search` (different `rrf_k`, a reranker, more candidates) and re-run this cell to see if the metrics actually improved.

In [ ]:
def evaluate_retriever(retrieve_fn, eval_set: list[dict], k: int) -> dict:
    per_query = []
    for item in eval_set:
        ranked = retrieve_fn(item["query"], k=k)
        relevant = item["relevant"]
        per_query.append({
            "query": item["query"],
            "ranked": ranked,
            "precision": precision_at_k(ranked, relevant, k),
            "recall": recall_at_k(ranked, relevant, k),
            "ndcg": ndcg_at_k(ranked, relevant, k),
        })
    n = len(per_query)
    averages = {
        "precision": sum(r["precision"] for r in per_query) / n,
        "recall": sum(r["recall"] for r in per_query) / n,
        "ndcg": sum(r["ndcg"] for r in per_query) / n,
    }
    return {"per_query": per_query, "averages": averages}


results = {name: evaluate_retriever(fn, EVAL_SET, K) for name, fn in RETRIEVERS.items()}

print(f"{'retriever':<10} {'P@' + str(K):>8} {'R@' + str(K):>8} {'NDCG@' + str(K):>8}")
for name, res in results.items():
    avg = res["averages"]
    print(f"{name:<10} {avg['precision']:>8.3f} {avg['recall']:>8.3f} {avg['ndcg']:>8.3f}")

### Per-query breakdown

Averages can hide the interesting part: *which* queries a retriever gets wrong. Print each query's NDCG@k per retriever, plus the chunks it actually returned, to see where dense and sparse diverge (this is the same disagreement `2_hybrid_rag/2_hybrid_search_rrf.ipynb` shows qualitatively - here it's quantified).

In [ ]:
for i, item in enumerate(EVAL_SET):
    print(f"[{i}] {item['query']!r}  (relevant={sorted(item['relevant'])})")
    for name, res in results.items():
        pq = res["per_query"][i]
        print(f"    {name:<8} ndcg={pq['ndcg']:.3f}  ranked={pq['ranked']}")
    print()

## Recap

- Retrieval quality is measured against a **labeled query set** - queries paired with the chunk IDs that should be retrieved for them.
- **Precision@k** catches noisy/irrelevant results; **Recall@k** catches missed relevant chunks; **NDCG@k** additionally rewards ranking relevant chunks *higher*.
- All three reduce to simple functions over a ranked ID list and a ground-truth set - no extra infrastructure beyond the retrievers already built in `2_hybrid_rag`.
- `evaluate_retriever` is retriever-agnostic: point it at `dense_search`, `sparse_search`, `hybrid_search`, or any future retriever (e.g. one with a cross-encoder reranker) to compare them on the same footing.
- This demo's eval set is small (14 queries, 18 chunks) for illustration - one query (the Deka LLM model categories) even has multiple relevant chunks, which is where precision/recall diverge most clearly. In practice, build a real eval set from representative user queries with careful relevance labeling - the metrics are only as trustworthy as the ground truth behind them.